In [ ]:
#调用集团满血版ds
from openai import OpenAI

# 初始化客户端，指向你的私有服务
client = OpenAI(
    base_url="http://10.231.4.6:30109/CIDC-ECSO-109/inference-proxy/792563b5-943d-434e-95f2-b9c94d5719c5/aiops-1421977318167711744/deepseek-v3-0324/service/8080/v1",
    api_key="FoLWIyIXgobLLNWxFkQS2VjCsPNMCsgk1tfelRqHy-w"  # 这就是 Bearer token
)

# 调用 chat completions
response = client.chat.completions.create(
    model="deepseek-v3-0324",
    messages=[
        {"role": "user", "content": "K8S是做微服务管理吗"}
    ],
    max_tokens=5000
)

# 打印回复内容
print(response.choices[0].message.content)


## OpenSearch Dashboard  API接口取数

In [1]:
# ========================================
#  第一步：填写用户名和密码
# ========================================
OS_USER = "sylvanli_rd"              # <-- 在这里填你的用户名
OS_PASSWORD = "94&ba!Rd3e3b03Slyan2"   # <-- 在这里填你的密码

print(f"用户名: {OS_USER}")
print("密码已设置")

用户名: sylvanli_rd
密码已设置


In [2]:
# ========================================
#  第三步：运行这个 cell 执行 SQL 查询
# ========================================
import http.client
import ssl
import json
import base64

# SSL 配置
ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
ctx.load_default_certs()

# ============================================
#  查询 Fortinet 相关数据（推荐方式）
#  用 _all + WHERE dtype 过滤
# ============================================
sql_query = "SELECT * FROM _all WHERE dtype = 'fortisandbox' LIMIT 20"
print(f"执行 SQL: {sql_query}")

# ============================================
#  如果你想查其他数据类型，修改 dtype 值：
#   - dtype = 'av-engine'          (防病毒日志)
#   - dtype = 'fortisandbox'       (Fortinet 沙箱日志)
#   - LIKE 模糊匹配：dtype LIKE '%forti%' (所有 Fortinet 相关)
# ============================================
# sql_query = "SELECT * FROM _all WHERE dtype LIKE '%forti%' LIMIT 20"
# print(f"执行 SQL: {sql_query}")

# 准备请求数据
data = json.dumps({"query": sql_query})

# 构造 Basic 认证
auth_str = f"{OS_USER}:{OS_PASSWORD}"
auth_b64 = base64.b64encode(auth_str.encode()).decode()

# 发送 POST 请求到 SQL 接口
conn = http.client.HTTPSConnection("192.168.100.45", 9200, timeout=60, context=ctx)
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Basic {auth_b64}"
}
conn.request("POST", "/_plugins/_sql", body=data, headers=headers)
response = conn.getresponse()

print(f"状态码: {response.status}")
result = json.loads(response.read().decode())
print(json.dumps(result, indent=2, ensure_ascii=False))

conn.close()

执行 SQL: SELECT * FROM _all WHERE dtype = 'fortisandbox' LIMIT 20
状态码: 200
{
  "schema": [
    {
      "name": "srcuuid",
      "type": "keyword"
    },
    {
      "name": "@parsing_timestamp",
      "type": "timestamp"
    },
    {
      "name": "reason",
      "type": "keyword"
    },
    {
      "name": "references",
      "type": "nested"
    },
    {
      "name": "_collect_timestamp",
      "type": "timestamp"
    },
    {
      "name": "httpmethod",
      "type": "keyword"
    },
    {
      "name": "rule_number",
      "type": "long"
    },
    {
      "name": "sessionid",
      "type": "keyword"
    },
    {
      "name": "observer",
      "type": "object"
    },
    {
      "name": "hostname",
      "type": "text"
    },
    {
      "name": "audit_rest_request_path",
      "type": "text"
    },
    {
      "name": "application_usage_totals",
      "type": "object"
    },
    {
      "name": "sourcetype",
      "type": "text"
    },
    {
      "name": "dstip",
      "type": "

In [3]:
# ========================================
# 第三步：运行这个 cell 执行 PPL 查询
# ========================================
import http.client
import ssl
import json
import base64

# SSL 配置
ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
ctx.load_default_certs()

# PPL 查询语句
ppl_query = """search source=`log_g*_fortigate_firewall-*` 
| where (subtype='system' and action='login' and status='failed') OR (subtype='vpn' and msg='SSL user failed to logged in') 
| where reason != 'ip_blocked' 
| where srcip != '218.92.0.39' 
| eval attack_src = if(isnotnull(srcip), srcip, remip) 
| fields @timestamp_cst, user, attack_src, srcip, remip, action, reason, msg, subtype, @gid, devname 
| sort - @timestamp_cst
| head 10"""

# 构造 Basic 认证
auth_str = f"{OS_USER}:{OS_PASSWORD}"
auth_b64 = base64.b64encode(auth_str.encode()).decode()

# 发送 POST 请求到 PPL 接口
conn = http.client.HTTPSConnection("192.168.100.45", 9200, timeout=60, context=ctx)
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Basic {auth_b64}"
}

# PPL 请求体
data = json.dumps({
    "query": ppl_query,
    "format": "json"  # 返回 JSON 格式
})

conn.request("POST", "/_opendistro/_ppl", body=data, headers=headers)
response = conn.getresponse()

print(f"状态码: {response.status}")
result = json.loads(response.read().decode())
print(json.dumps(result, indent=2, ensure_ascii=False))

conn.close()

状态码: 400
{
  "error": "no handler found for uri [/_opendistro/_ppl] and method [POST]"
}


In [4]:
# ========================================
# 第三步：运行这个 cell 执行 PPL 查询
# ========================================
import http.client
import ssl
import json
import base64

# SSL 配置
ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
ctx.load_default_certs()

# PPL 查询语句
ppl_query = """search source=`log_g*_fortigate_firewall-*` 
| where (subtype='system' and action='login' and status='failed') OR (subtype='vpn' and msg='SSL user failed to logged in') 
| where reason != 'ip_blocked' 
| where srcip != '218.92.0.39' 
| eval attack_src = if(isnotnull(srcip), srcip, remip) 
| fields @timestamp_cst, user, attack_src, srcip, remip, action, reason, msg, subtype, @gid, devname 
| sort - @timestamp_cst
| head 2"""

# 构造 Basic 认证
auth_str = f"{OS_USER}:{OS_PASSWORD}"
auth_b64 = base64.b64encode(auth_str.encode()).decode()

# 发送 POST 请求到 PPL 接口
conn = http.client.HTTPSConnection("192.168.100.45", 9200, timeout=60, context=ctx)
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Basic {auth_b64}"
}

# PPL 请求体
data = json.dumps({
    "query": ppl_query,
    "format": "json"  # 返回 JSON 格式
})

conn.request("POST", "/_opendistro/_ppl", body=data, headers=headers)
response = conn.getresponse()

print(f"状态码: {response.status}")
result = json.loads(response.read().decode())
print(json.dumps(result, indent=2, ensure_ascii=False))

conn.close()

状态码: 400
{
  "error": "no handler found for uri [/_opendistro/_ppl] and method [POST]"
}


## 日志压缩

In [12]:
# LongLLMLingua JSON日志压缩测试
from llmlingua import PromptCompressor

llm_lingua = PromptCompressor()

# 构造模拟的暴力破解日志数据（30条）
sample_logs = [
    {"timestamp": "2024-01-15 22:15:33", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 5},
    {"timestamp": "2024-01-15 22:15:34", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 3},
    {"timestamp": "2024-01-15 22:16:01", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 10},
    {"timestamp": "2024-01-15 22:16:15", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 7},
    {"timestamp": "2024-01-15 22:17:00", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 2},
    {"timestamp": "2024-01-15 22:18:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:19:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 15},
    {"timestamp": "2024-01-15 22:20:10", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "ubuntu", "fail_count": 4},
    {"timestamp": "2024-01-15 22:21:33", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 20},
    {"timestamp": "2024-01-15 22:22:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 12},
    {"timestamp": "2024-01-15 22:23:15", "src_ip": "5.5.5.5", "dst_ip": "192.168.1.30", "port": 5432, "event": "PostgreSQL连接失败", "username": "postgres", "fail_count": 6},
    {"timestamp": "2024-01-15 22:24:30", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 9},
    {"timestamp": "2024-01-15 22:25:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 25},
    {"timestamp": "2024-01-15 22:26:11", "src_ip": "6.6.6.6", "dst_ip": "192.168.1.35", "port": 22, "event": "SSH登录失败", "username": "oracle", "fail_count": 3},
    {"timestamp": "2024-01-15 22:27:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 18},
    {"timestamp": "2024-01-15 22:28:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 30},
    {"timestamp": "2024-01-15 22:29:10", "src_ip": "7.7.7.7", "dst_ip": "192.168.1.40", "port": 22, "event": "SSH登录失败", "username": "centos", "fail_count": 5},
    {"timestamp": "2024-01-15 22:30:00", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 22},
    {"timestamp": "2024-01-15 22:31:15", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:32:30", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 50},
]

# 将JSON日志转为文本格式
log_text = "\n".join([str(log) for log in sample_logs])

print("=" * 60)
print("原始日志信息:")
print("=" * 60)
print(f"日志条数: {len(sample_logs)}")
print(f"原始文本长度: {len(log_text)} 字符")
print("-" * 60)
print(log_text)
print("=" * 60)

# 执行压缩
question = "分析这次暴力破解攻击的特征，并给出防御建议"

compressed_prompt = llm_lingua.compress_prompt(
    prompt_list=[log_text],
    question=question,
    rate=0.5,  # 压缩到50%
    condition_in_question="after_condition",
    reorder_context="sort",
    dynamic_context_compression_ratio=0.3,
)

print("\n压缩后:")
print("=" * 60)
print(f"压缩后文本长度: {len(compressed_prompt['compressed_prompt'])} 字符")
print(f"压缩率: {len(compressed_prompt['compressed_prompt']) / len(log_text) * 100:.1f}%")
print("-" * 60)
print(compressed_prompt['compressed_prompt'])
print("=" * 60)

print("\n原始和压缩后的对比:")
print(f"原始长度: {len(log_text)}")
print(f"压缩后长度: {len(compressed_prompt['compressed_prompt'])}")
print(f"压缩比: {len(log_text) / len(compressed_prompt['compressed_prompt']):.2f}x")

ModuleNotFoundError: No module named 'llmlingua'

In [13]:
import sys
import os

print("当前 Python:", sys.executable)
print("\n当前环境路径:", os.environ.get('VIRTUAL_ENV', 'Not in venv'))

# 尝试直接安装到当前环境
import subprocess
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'llmlingua'], 
                       capture_output=True, text=True)
print("\n安装输出:", result.stdout)
print("安装错误:", result.stderr)

当前 Python: /usr/local/bin/python

当前环境路径: Not in venv

安装输出: Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.9 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━ 30.7/40.9 kB 992.0 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 919.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.6 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.6 MB 997.5 kB/s eta 0:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/10.6 MB 756.0 kB/s eta 0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/10.6 MB 711.3 kB/s eta 0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/10.6 MB 733.8 kB/s eta 0:00:15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.1/10.6 MB 697.5 kB/s eta 0:00:16
   ╸━━━━━━

In [3]:
import sys
print(sys.executable)  # 应该显示 ysrzfx 虚拟环境路径


/usr/local/bin/python


In [22]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name="/app/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots/5f0c82792b7ea14c6484e015b6a072009496b7f2"
)

sample_logs = [
    {"timestamp": "2024-01-15 22:15:33", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 5},
    {"timestamp": "2024-01-15 22:15:34", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 3},
    {"timestamp": "2024-01-15 22:16:01", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 10},
    {"timestamp": "2024-01-15 22:16:15", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 7},
    {"timestamp": "2024-01-15 22:17:00", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 2},
    {"timestamp": "2024-01-15 22:18:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:19:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 15},
    {"timestamp": "2024-01-15 22:20:10", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "ubuntu", "fail_count": 4},
    {"timestamp": "2024-01-15 22:21:33", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 20},
    {"timestamp": "2024-01-15 22:22:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 12},
    {"timestamp": "2024-01-15 22:23:15", "src_ip": "5.5.5.5", "dst_ip": "192.168.1.30", "port": 5432, "event": "PostgreSQL连接失败", "username": "postgres", "fail_count": 6},
    {"timestamp": "2024-01-15 22:24:30", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 9},
    {"timestamp": "2024-01-15 22:25:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 25},
    {"timestamp": "2024-01-15 22:26:11", "src_ip": "6.6.6.6", "dst_ip": "192.168.1.35", "port": 22, "event": "SSH登录失败", "username": "oracle", "fail_count": 3},
    {"timestamp": "2024-01-15 22:27:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 18},
    {"timestamp": "2024-01-15 22:28:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 30},
    {"timestamp": "2024-01-15 22:29:10", "src_ip": "7.7.7.7", "dst_ip": "192.168.1.40", "port": 22, "event": "SSH登录失败", "username": "centos", "fail_count": 5},
    {"timestamp": "2024-01-15 22:30:00", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 22},
    {"timestamp": "2024-01-15 22:31:15", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:32:30", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 50},
]

log_text = "\n".join([str(log) for log in sample_logs])

print("=" * 60)
print("原始日志信息:")
print("=" * 60)
print(f"日志条数: {len(sample_logs)}")
print(f"原始文本长度: {len(log_text)} 字符")
print("-" * 60)
print(log_text[:500], "...")
print("=" * 60)

question = "分析这次暴力破解攻击的特征，并给出防御建议"

compressed_prompt = llm_lingua.compress_prompt(
    prompt_list=[log_text],
    question=question,
    rate=0.5,
)

print("\n压缩后:")
print("=" * 60)
print(f"压缩后文本长度: {len(compressed_prompt['compressed_prompt'])} 字符")
print(f"压缩率: {len(compressed_prompt['compressed_prompt']) / len(log_text) * 100:.1f}%")
print("-" * 60)
print(compressed_prompt['compressed_prompt'])
print("=" * 60)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/app/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots/5f0c82792b7ea14c6484e015b6a072009496b7f2'. Use `repo_type` argument if needed.

In [27]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/home/sylvanli/ysrzfx/hf_cache'

from llmlingua import PromptCompressor

llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank"
)

print("模型加载成功!")

sample_logs = [
    {"timestamp": "2024-01-15 22:15:33", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 5},
    {"timestamp": "2024-01-15 22:15:34", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 3},
    {"timestamp": "2024-01-15 22:16:01", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 10},
    {"timestamp": "2024-01-15 22:16:15", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 7},
    {"timestamp": "2024-01-15 22:17:00", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 2},
    {"timestamp": "2024-01-15 22:18:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:19:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 15},
    {"timestamp": "2024-01-15 22:20:10", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "ubuntu", "fail_count": 4},
    {"timestamp": "2024-01-15 22:21:33", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 20},
    {"timestamp": "2024-01-15 22:22:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 12},
    {"timestamp": "2024-01-15 22:23:15", "src_ip": "5.5.5.5", "dst_ip": "192.168.1.30", "port": 5432, "event": "PostgreSQL连接失败", "username": "postgres", "fail_count": 6},
    {"timestamp": "2024-01-15 22:24:30", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 9},
    {"timestamp": "2024-01-15 22:25:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 25},
    {"timestamp": "2024-01-15 22:26:11", "src_ip": "6.6.6.6", "dst_ip": "192.168.1.35", "port": 22, "event": "SSH登录失败", "username": "oracle", "fail_count": 3},
    {"timestamp": "2024-01-15 22:27:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 18},
    {"timestamp": "2024-01-15 22:28:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 30},
    {"timestamp": "2024-01-15 22:29:10", "src_ip": "7.7.7.7", "dst_ip": "192.168.1.40", "port": 22, "event": "SSH登录失败", "username": "centos", "fail_count": 5},
    {"timestamp": "2024-01-15 22:30:00", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 22},
    {"timestamp": "2024-01-15 22:31:15", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:32:30", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 50},
]

log_text = "\n".join([str(log) for log in sample_logs])

print("=" * 60)
print("原始日志信息:")
print("=" * 60)
print(f"日志条数: {len(sample_logs)}")
print(f"原始文本长度: {len(log_text)} 字符")
print("-" * 60)
print(log_text[:500], "...")
print("=" * 60)

question = "分析这次暴力破解攻击的特征，并给出防御建议"

compressed_prompt = llm_lingua.compress_prompt(
    prompt_list=[log_text],
    question=question,
    rate=0.5,
)

print("\n压缩后:")
print("=" * 60)
print(f"压缩后文本长度: {len(compressed_prompt['compressed_prompt'])} 字符")
print(f"压缩率: {len(compressed_prompt['compressed_prompt']) / len(log_text) * 100:.1f}%")
print("-" * 60)
print(compressed_prompt['compressed_prompt'])
print("=" * 60)

'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank' is the correct path to a directory containing a config.json file

In [26]:
import os

cache_dir = "/home/sylvanli/ysrzfx/hf_cache"
print("HF cache 存在:", os.path.exists(cache_dir))

if os.path.exists(cache_dir):
    print("目录内容:", os.listdir(cache_dir))

HF cache 存在: True
目录内容: ['.locks', 'models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank', 'CACHEDIR.TAG', 'llmlingua-bert']


In [28]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/home/sylvanli/ysrzfx/hf_cache'

from transformers import AutoTokenizer, AutoModel

model_path = '/home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots'

# 找实际路径
import subprocess
result = subprocess.run(['find', model_path, '-name', 'config.json'], capture_output=True, text=True)
actual_path = os.path.dirname(result.stdout.strip())
print(f"实际模型路径: {actual_path}")

tokenizer = AutoTokenizer.from_pretrained(actual_path)
model = AutoModel.from_pretrained(actual_path)
print("模型加载成功!")

实际模型路径: /home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots/5f0c82792b7ea14c6484e015b6a072009496b7f2


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9747.75it/s]
[transformers] BertModel LOAD REPORT from: /home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots/5f0c82792b7ea14c6484e015b6a072009496b7f2
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


模型加载成功!


In [2]:
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")
# os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# os.environ['HF_HOME'] = '/home/sylvanli/ysrzfx/hf_cache'

# 修改这里：使用 /tmp 目录，通常权限最宽松
os.environ['HF_HOME'] = '/tmp/hf_cache' 
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 确保目录存在
os.makedirs('/tmp/hf_cache', exist_ok=True)

# ... 后面的代码保持不变 ...

import subprocess
model_path = '/home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots'
result = subprocess.run(['find', model_path, '-name', 'config.json'], capture_output=True, text=True)
actual_path = os.path.dirname(result.stdout.strip())

from llmlingua import PromptCompressor

# llm_lingua = PromptCompressor(
#     model_name=actual_path,
#     device_map="cpu",
#     model_config={"is_causal_lm": True} # 👈 强制指定为因果语言模型（生成模型）
# )
# 尝试直接使用模型ID，而不是本地路径
llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank", 
    device_map="cpu"
)
print("模型加载成功!")

sample_logs = [
    {"timestamp": "2024-01-15 22:15:33", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 5},
    {"timestamp": "2024-01-15 22:15:34", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 3},
    {"timestamp": "2024-01-15 22:16:01", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 10},
    {"timestamp": "2024-01-15 22:16:15", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 7},
    {"timestamp": "2024-01-15 22:17:00", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 2},
    {"timestamp": "2024-01-15 22:18:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:19:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 15},
    {"timestamp": "2024-01-15 22:20:10", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "ubuntu", "fail_count": 4},
    {"timestamp": "2024-01-15 22:21:33", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 20},
    {"timestamp": "2024-01-15 22:22:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 12},
    {"timestamp": "2024-01-15 22:23:15", "src_ip": "5.5.5.5", "dst_ip": "192.168.1.30", "port": 5432, "event": "PostgreSQL连接失败", "username": "postgres", "fail_count": 6},
    {"timestamp": "2024-01-15 22:24:30", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 9},
    {"timestamp": "2024-01-15 22:25:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 25},
    {"timestamp": "2024-01-15 22:26:11", "src_ip": "6.6.6.6", "dst_ip": "192.168.1.35", "port": 22, "event": "SSH登录失败", "username": "oracle", "fail_count": 3},
    {"timestamp": "2024-01-15 22:27:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 18},
    {"timestamp": "2024-01-15 22:28:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 30},
    {"timestamp": "2024-01-15 22:29:10", "src_ip": "7.7.7.7", "dst_ip": "192.168.1.40", "port": 22, "event": "SSH登录失败", "username": "centos", "fail_count": 5},
    {"timestamp": "2024-01-15 22:30:00", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 22},
    {"timestamp": "2024-01-15 22:31:15", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:32:30", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 50},
]

log_text = "\n".join([str(log) for log in sample_logs])

print("=" * 60)
print("原始日志信息:")
print("=" * 60)
print(f"日志条数: {len(sample_logs)}")
print(f"原始文本长度: {len(log_text)} 字符")
print("-" * 60)
print(log_text[:500], "...")
print("=" * 60)

question = "分析这次暴力破解攻击的特征，并给出防御建议"

# 正确的 API 参数
compressed_prompt = llm_lingua.compress_prompt(
    context=log_text,
    question=question,
    rate=0.5,
)

print("\n压缩后:")
print("=" * 60)
print(f"压缩后文本长度: {len(compressed_prompt)} 字符")
print(f"压缩率: {len(compressed_prompt) / len(log_text) * 100:.1f}%")
print("-" * 60)
print(compressed_prompt)
print("=" * 60)

OSError: PermissionError at /home/sylvanli/ysrzfx/hf_cache/hub when downloading microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank. Check cache directory permissions. Common causes: 1) another user is downloading the same model (please wait); 2) a previous download was canceled and the lock file needs manual removal.

### transformers + tiktoken 实现压缩测试：

In [2]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['HF_HOME'] = '/home/sylvanli/ysrzfx/hf_cache'

import torch
from transformers import AutoTokenizer, AutoModel
import tiktoken

# 加载模型
import subprocess
model_path = '/home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots'
result = subprocess.run(['find', model_path, '-name', 'config.json'], capture_output=True, text=True)
actual_path = os.path.dirname(result.stdout.strip())

tokenizer = AutoTokenizer.from_pretrained(actual_path)
model = AutoModel.from_pretrained(actual_path)
model.eval()

print("模型加载成功!")

# 测试数据
sample_logs = [
    {"timestamp": "2024-01-15 22:15:33", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 5},
    {"timestamp": "2024-01-15 22:15:34", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 3},
    {"timestamp": "2024-01-15 22:16:01", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 10},
    {"timestamp": "2024-01-15 22:16:15", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 7},
    {"timestamp": "2024-01-15 22:17:00", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 2},
    {"timestamp": "2024-01-15 22:18:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:19:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 15},
    {"timestamp": "2024-01-15 22:20:10", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "ubuntu", "fail_count": 4},
    {"timestamp": "2024-01-15 22:21:33", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 20},
    {"timestamp": "2024-01-15 22:22:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 12},
    {"timestamp": "2024-01-15 22:23:15", "src_ip": "5.5.5.5", "dst_ip": "192.168.1.30", "port": 5432, "event": "PostgreSQL连接失败", "username": "postgres", "fail_count": 6},
    {"timestamp": "2024-01-15 22:24:30", "src_ip": "3.3.3.3", "dst_ip": "192.168.1.20", "port": 3389, "event": "RDP登录失败", "username": "Administrator", "fail_count": 9},
    {"timestamp": "2024-01-15 22:25:00", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 25},
    {"timestamp": "2024-01-15 22:26:11", "src_ip": "6.6.6.6", "dst_ip": "192.168.1.35", "port": 22, "event": "SSH登录失败", "username": "oracle", "fail_count": 3},
    {"timestamp": "2024-01-15 22:27:22", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "admin", "fail_count": 18},
    {"timestamp": "2024-01-15 22:28:45", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "test", "fail_count": 30},
    {"timestamp": "2024-01-15 22:29:10", "src_ip": "7.7.7.7", "dst_ip": "192.168.1.40", "port": 22, "event": "SSH登录失败", "username": "centos", "fail_count": 5},
    {"timestamp": "2024-01-15 22:30:00", "src_ip": "2.2.2.2", "dst_ip": "192.168.1.15", "port": 3306, "event": "MySQL连接失败", "username": "root", "fail_count": 22},
    {"timestamp": "2024-01-15 22:31:15", "src_ip": "4.4.4.4", "dst_ip": "192.168.1.25", "port": 22, "event": "SSH登录失败", "username": "admin", "fail_count": 8},
    {"timestamp": "2024-01-15 22:32:30", "src_ip": "1.1.1.1", "dst_ip": "192.168.1.10", "port": 22, "event": "SSH登录失败", "username": "root", "fail_count": 50},
]

# 计算每条日志的 BERT embedding
def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0].numpy()

# 计算句子重要性分数
def calculate_importance_scores(logs, question):
    question_emb = get_sentence_embedding(question)
    scores = []
    for log in logs:
        log_emb = get_sentence_embedding(str(log))
        score = float(torch.cosine_similarity(
            torch.tensor(question_emb).unsqueeze(0),
            torch.tensor(log_emb).unsqueeze(0)
        ))
        scores.append(score)
    return scores

# 简单压缩：按分数排序，保留 top N
def simple_compress(logs, question, keep_ratio=0.5):
    scores = calculate_importance_scores(logs, question)
    indexed_logs = list(zip(scores, logs))
    indexed_logs.sort(reverse=True, key=lambda x: x[0])
    keep_count = int(len(logs) * keep_ratio)
    return [str(log) for _, log in indexed_logs[:keep_count]]

log_text = "\n".join([str(log) for log in sample_logs])
enc = tiktoken.get_encoding("cl100k_base")
original_tokens = len(enc.encode(log_text))

print("=" * 60)
print("原始日志信息:")
print("=" * 60)
print(f"日志条数: {len(sample_logs)}")
print(f"原始字符数: {len(log_text)}")
print(f"原始 Token 数: {original_tokens}")
print("-" * 60)
print(log_text[:500], "...")
print("=" * 60)

question = "分析这次暴力破解攻击的特征，并给出防御建议"

print("\n正在计算每条日志的重要性分数...")

# 压缩
compressed_logs = simple_compress(sample_logs, question, keep_ratio=0.5)
compressed_text = "\n".join(compressed_logs)
compressed_tokens = len(enc.encode(compressed_text))

print("\n压缩后:")
print("=" * 60)
print(f"保留条数: {len(compressed_logs)}/{len(sample_logs)}")
print(f"压缩后字符数: {len(compressed_text)}")
print(f"压缩后 Token 数: {compressed_tokens}")
print(f"压缩率: {compressed_tokens/original_tokens*100:.1f}%")
print("-" * 60)
print(compressed_text)
print("=" * 60)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9454.63it/s]
[transformers] BertModel LOAD REPORT from: /home/sylvanli/ysrzfx/hf_cache/models--microsoft--llmlingua-2-bert-base-multilingual-cased-meetingbank/snapshots/5f0c82792b7ea14c6484e015b6a072009496b7f2
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


模型加载成功!
原始日志信息:
日志条数: 20
原始字符数: 3134
原始 Token 数: 1412
------------------------------------------------------------
{'timestamp': '2024-01-15 22:15:33', 'src_ip': '1.1.1.1', 'dst_ip': '192.168.1.10', 'port': 22, 'event': 'SSH登录失败', 'username': 'root', 'fail_count': 5}
{'timestamp': '2024-01-15 22:15:34', 'src_ip': '1.1.1.1', 'dst_ip': '192.168.1.10', 'port': 22, 'event': 'SSH登录失败', 'username': 'admin', 'fail_count': 3}
{'timestamp': '2024-01-15 22:16:01', 'src_ip': '2.2.2.2', 'dst_ip': '192.168.1.15', 'port': 3306, 'event': 'MySQL连接失败', 'username': 'root', 'fail_count': 10}
{'timestamp': '2024-01-15 22:16:15' ...

正在计算每条日志的重要性分数...

压缩后:
保留条数: 10/20
压缩后字符数: 1561
压缩后 Token 数: 705
压缩率: 49.9%
------------------------------------------------------------
{'timestamp': '2024-01-15 22:29:10', 'src_ip': '7.7.7.7', 'dst_ip': '192.168.1.40', 'port': 22, 'event': 'SSH登录失败', 'username': 'centos', 'fail_count': 5}
{'timestamp': '2024-01-15 22:26:11', 'src_ip': '6.6.6.6', 'dst_ip': '192.168.1.35', 'p